# Week 2 Day 3: returning a payout for review
Andras Nemes. First challenge from `labs/week2day3.ipynb`.

A handler rejects a **proposed payout**, explains why in a note, and the agent reconsiders the claim. Rejection is not an automatic decline. Approval goes to payment; rejection adds a human message and loops back through the agent and decision nodes. A revised payout over $1,000 needs a new sign-off.

Run with the repository's `.venv` kernel. The default run performs offline checks with scripted model responses and no external services. The optional live section uses `OPENROUTER_API_KEY` and `DATABASE_URL` from the root `.env`, the session pooler from the lab, and its own `lab_day3_rejection` schema. The second-computer stretch challenge is excluded.

In [ ]:
import json
import sys
from collections.abc import Callable
from pathlib import Path
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import Runnable, RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, tool
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field, StrictBool, model_validator

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "labs/week2day3.ipynb").is_file())
if str(ROOT / "labs") not in sys.path:
    sys.path.insert(0, str(ROOT / "labs"))
from week2day1_payout import Benefit, Line, Payout, work_out_payout

DATA = ROOT / "data"
claims = {c["claim_id"]: c for c in json.loads((DATA / "claims.json").read_text(encoding="utf-8"))}
policies = {p["policy_id"]: p for p in json.loads((DATA / "policies.json").read_text(encoding="utf-8"))}
APPROVAL_LIMIT = json.loads((DATA / "company.json").read_text(encoding="utf-8"))["payout_approval_limit"]


@tool
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Calculate an approved payout; group pairs and sets as one item, and count separate items using quantity."""
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

## State and handler response
`approved` controls routing. `note` explains what to revisit and is appended to the conversation on rejection, so the model sees it on its next turn. The checkpointer saves that conversation on the same claim thread. Each new decision clears the previous sign-off.

In [ ]:
class ClaimState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]
    claim_id: str
    decision: str
    payout: float
    rationale: str
    approved: bool
    approved_by: str
    note: str


class ClaimDecision(BaseModel):
    rationale: str = Field(description="Explain the decision and relevant policy sections in 2-3 sentences")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(ge=0, description="US dollars; zero for a decline or referral")


class HandlerAnswer(BaseModel):
    handler: str = Field(min_length=1)
    approved: StrictBool
    note: str

    @model_validator(mode="after")
    def validate_feedback(self) -> "HandlerAnswer":
        if not self.handler.strip() or (not self.approved and not self.note.strip()):
            raise ValueError("A handler name and a rejection note are required")
        return self


def require_payable(state: ClaimState) -> None:
    """Protect the payment node even if called directly."""
    if state["decision"] != "approve" or state["payout"] <= 0:
        raise ValueError("Only an approval with a positive payout can be paid")
    if state["payout"] > APPROVAL_LIMIT and not (state.get("approved") and state.get("approved_by")):
        raise ValueError("A payout over the limit needs sign-off")


def review(graph: CompiledStateGraph, config: RunnableConfig, answer: HandlerAnswer) -> ClaimState:
    """Resume the pending review on the same thread."""
    if graph.get_state(config).next != ("approval",):
        raise ValueError("This thread is not waiting for a handler")
    return graph.invoke(Command(resume=answer.model_dump()), config, durability="sync")

In [ ]:
def build_graph(
    agent_model: Runnable,
    decision_model: Runnable,
    payout_tool: BaseTool,
    pay_node: Callable[[ClaimState], ClaimState],
    checkpointer: BaseCheckpointSaver,
    system: SystemMessage,
) -> CompiledStateGraph:
    """Build the lab graph with a conditional route after human review."""
    def agent(state: ClaimState) -> ClaimState:
        return {"messages": [agent_model.invoke([system, *state["messages"]])]}

    def decide(state: ClaimState) -> ClaimState:
        result = decision_model.invoke([system, *state["messages"], HumanMessage("Record your decision on this claim.")])
        return {"decision": result.decision, "payout": result.payout, "rationale": result.rationale,
                "approved": False, "approved_by": "", "note": ""}

    def approval(state: ClaimState) -> ClaimState:
        answer = HandlerAnswer.model_validate(interrupt({
            "claim_id": state["claim_id"], "payout": state["payout"], "rationale": state["rationale"],
        }))
        update: ClaimState = {"approved": answer.approved, "approved_by": answer.handler if answer.approved else "",
                              "note": answer.note}
        if not answer.approved:
            update["messages"] = [HumanMessage(
                f"Handler {answer.handler} rejected the proposed payout of ${state['payout']:,.2f}. "
                f"Proposed rationale: {state['rationale']}\nFeedback: {answer.note}\n"
                "Reconsider this claim against the policy, recalculate as needed, and explain your revised decision."
            )]
        return update

    def after_agent(state: ClaimState) -> str:
        return "tools" if state["messages"][-1].tool_calls else "decide"

    def after_decide(state: ClaimState) -> str:
        if state["decision"] != "approve":
            return END
        return "approval" if state["payout"] > APPROVAL_LIMIT else "pay"

    def after_approval(state: ClaimState) -> str:
        return "pay" if state["approved"] else "agent"

    builder = StateGraph(ClaimState)
    for name, node in [("agent", agent), ("tools", ToolNode([payout_tool])), ("decide", decide),
                       ("approval", approval), ("pay", pay_node)]:
        builder.add_node(name, node)
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", after_agent, ["tools", "decide"])
    builder.add_edge("tools", "agent")
    builder.add_conditional_edges("decide", after_decide, ["approval", "pay", END])
    builder.add_conditional_edges("approval", after_approval, ["pay", "agent"])
    builder.add_edge("pay", END)
    return builder.compile(checkpointer=checkpointer)

## Offline checks
Scripted responses make the rejection path reproducible even when a real model gets its first proposal right. These checks exercise the compiled graph, including two rejections, feedback delivery, a fresh sign-off, a revised amount below the limit, and revised declines/referrals. They also check that blank rejection notes are refused and payment cannot bypass sign-off. They do not test a live model or Postgres connection.

In [ ]:
def scripted_graph(answers: list[ClaimDecision]) -> tuple[CompiledStateGraph, list[list[BaseMessage]], list[tuple[float, str]]]:
    pending = iter(answers)
    seen: list[list[BaseMessage]] = []
    payments: list[tuple[float, str]] = []

    def model(messages: list[BaseMessage]) -> AIMessage:
        seen.append(messages)
        return AIMessage(content="Proposal ready for recording.")

    def decider(messages: list[BaseMessage]) -> ClaimDecision:
        return next(pending)

    def pay(state: ClaimState) -> ClaimState:
        require_payable(state)
        payments.append((state["payout"], state.get("approved_by", "")))
        return {}

    return build_graph(RunnableLambda(model), RunnableLambda(decider), calculate_payout, pay,
                       InMemorySaver(), SystemMessage("Offline workflow check.")), seen, payments


def proposal(payout: float, decision: str = "approve") -> ClaimDecision:
    return ClaimDecision(rationale="Scripted workflow fixture.", decision=decision, payout=payout)


def check_workflow() -> None:
    config: RunnableConfig = {"configurable": {"thread_id": "offline-CLM-1011"}}
    initial: ClaimState = {"claim_id": "CLM-1011", "messages": [HumanMessage("Review this claim.")]}
    graph, seen, payments = scripted_graph([proposal(1500), proposal(1250), proposal(1150)])
    graph.invoke(initial, config)
    for note in ["Check the suit and camera sets.", "Recheck the airline settlement."]:
        result = review(graph, config, HandlerAnswer(handler="Dana Whitfield", approved=False, note=note))
        assert not payments and graph.get_state(config).next == ("approval",)
        assert not result["approved"] and result["approved_by"] == ""
        assert any(note in message.content for message in seen[-1] if message.type == "human")
    result = review(graph, config, HandlerAnswer(handler="Dana Whitfield", approved=True, note="Calculation checked."))
    assert result["payout"] == 1150 and payments == [(1150, "Dana Whitfield")]
    assert not graph.get_state(config).next
    history = list(graph.get_state_history(config))
    assert any("Recheck the airline settlement." in str(s.values.get("messages", [])) for s in history)

    for outcome in [proposal(900), proposal(0, "decline"), proposal(0, "refer")]:
        graph, _, payments = scripted_graph([proposal(1500), outcome])
        graph.invoke(initial, config)
        review(graph, config, HandlerAnswer(handler="Dana Whitfield", approved=False, note="Reconsider coverage and amount."))
        assert not graph.get_state(config).next
        assert payments == ([(900, "")] if outcome.decision == "approve" else [])

    graph, _, payments = scripted_graph([proposal(1150)])
    graph.invoke(initial, config)
    review(graph, config, HandlerAnswer(handler="Dana Whitfield", approved=True, note=""))
    assert payments == [(1150, "Dana Whitfield")]
    try:
        HandlerAnswer(handler="Dana Whitfield", approved=False, note="   ")
    except ValueError:
        pass
    else:
        raise AssertionError("A rejection must carry feedback")
    try:
        require_payable({"decision": "approve", "payout": 1150, "approved": False})
    except ValueError:
        pass
    else:
        raise AssertionError("Payment bypassed sign-off")
    print("Offline workflow checks passed.")


check_workflow()

## Optional live run: CLM-1011
Set `RUN_LIVE = True` to connect and call the model. This uses the lab's payout tool and Supabase checkpointer, with a separate schema and a stable thread ID. Rerunning the start cell displays the existing state rather than deleting checkpoints or payments. Use this contribution's database objects only for this experiment.

A note identifies the issue to investigate; it does not force a specific answer. The notebook's $1,150 reference stays outside the model's system prompt. The initial proposal may already be correct; the rejection cell still demonstrates returning it for a further review.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    import os

    import psycopg
    from dotenv import load_dotenv
    from langchain_openai import ChatOpenAI
    from langgraph.checkpoint.postgres import PostgresSaver
    from psycopg.rows import dict_row

    load_dotenv(ROOT / ".env", override=True)
    connection = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, prepare_threshold=0,
                                 row_factory=dict_row, options="-c search_path=lab_day3_rejection")
    connection.execute("CREATE SCHEMA IF NOT EXISTS lab_day3_rejection")
    connection.execute("""CREATE TABLE IF NOT EXISTS payments (
        claim_id TEXT PRIMARY KEY, amount NUMERIC NOT NULL, approved_by TEXT,
        paid_at TIMESTAMPTZ DEFAULT now())""")
    saver = PostgresSaver(connection)
    saver.setup()

    def pay_live(state: ClaimState) -> ClaimState:
        require_payable(state)
        connection.execute(
            "INSERT INTO payments (claim_id, amount, approved_by) VALUES (%s, %s, %s) "
            "ON CONFLICT (claim_id) DO NOTHING",
            (state["claim_id"], state["payout"], state.get("approved_by")),
        )
        return {}

    instructions = """You are a claims handler at UnderCover Travel Insurance.
Decide the supplied claim using the policy wording and customer's policy record.
Approve covered claims with the required documents; decline claims outside cover;
refer claims with missing documents, conflicting evidence or unresolved wording.
Use calculate_payout for approvals, grouping pairs and sets as one item and passing
any payments by others. Explain the decision with the relevant policy sections.
When a handler returns a proposal, reconsider their feedback against the policy
and claim evidence, then give a revised decision and payout.
"""
    system = SystemMessage(instructions + (DATA / "policy_wording.md").read_text(encoding="utf-8"))
    model = ChatOpenAI(model=os.getenv("MODEL_NAME") or "openai/gpt-6-luna", reasoning_effort="low",
                       base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"],
                       timeout=120, max_retries=5)
    live_graph = build_graph(model.bind_tools([calculate_payout]), model.with_structured_output(ClaimDecision),
                             calculate_payout, pay_live, saver, system)
    live_config: RunnableConfig = {"configurable": {"thread_id": "day3-rejection-CLM-1011"}}

In [ ]:
if RUN_LIVE:
    snapshot = live_graph.get_state(live_config)
    if not snapshot.values:
        claim = claims["CLM-1011"]
        prompt = json.dumps({"policy": policies[claim["policy_id"]], "claim": claim}, indent=2)
        live_graph.invoke({"claim_id": claim["claim_id"], "messages": [HumanMessage(prompt)]},
                          live_config, durability="sync")
    snapshot = live_graph.get_state(live_config)
    print({key: snapshot.values.get(key) for key in ["decision", "payout", "rationale", "approved_by"]})
    print("Next:", snapshot.next)
    print(connection.execute("SELECT * FROM payments WHERE claim_id = %s", ("CLM-1011",)).fetchall())

In [ ]:
NOTE = (
    "Please recalculate under sections 5 and 10. Treat the jacket and trousers as one suit, "
    "and the camera body and lens as one set. Count the four shirts separately. "
    "Account for the airline's $350 settlement and the Premium deductible. Explain the calculation."
)

if RUN_LIVE and live_graph.get_state(live_config).next == ("approval",):
    revised = review(live_graph, live_config, HandlerAnswer(handler="Dana Whitfield", approved=False, note=NOTE))
    print({key: revised[key] for key in ["decision", "payout", "rationale"]})
    assert not connection.execute("SELECT 1 FROM payments WHERE claim_id = %s", ("CLM-1011",)).fetchone()
    print("Next:", live_graph.get_state(live_config).next)

Inspect the revised reasoning before signing off. The reference payout is **$1,150**: suitcase $260, suit capped at $300, camera set capped at $300, shoes $230, four shirts $340 and cufflinks $120 total $1,550; subtract $350 from the airline and the $50 deductible. It remains above the approval limit.

If the revision is still wrong, rerun the rejection cell with a more specific note. Set `APPROVE_REVISED = True` only when satisfied. An approval concerns the current proposal; every later revision must obtain its own sign-off.

In [ ]:
APPROVE_REVISED = False

if RUN_LIVE and APPROVE_REVISED:
    final = review(live_graph, live_config,
                   HandlerAnswer(handler="Dana Whitfield", approved=True, note="Calculation reviewed."))
    print("Final:", final["decision"], final["payout"], final["approved_by"])
    print(connection.execute("SELECT * FROM payments WHERE claim_id = %s", ("CLM-1011",)).fetchall())

## Takeaway
The approval flag controls permission to pay; the note supplies context for reconsideration. Conditional edges enforce the loop, while the existing message reducer preserves feedback. Postgres checkpoints retain the thread across restarts, and the payment table's primary key prevents duplicate inserts if the payment node is retried. Code before `interrupt()` may run again, so payment remains in a separate node.

The offline checks verify workflow behavior. A live run is optional and its outcome must be inspected; no model accuracy or database recovery result is claimed by the offline checks.

In [ ]:
if RUN_LIVE:
    connection.close()